<a href="https://colab.research.google.com/github/IzaakGagnon/Interval-Estimation-of-Phi/blob/main/Interval_Estimation_for_Integrated_Information.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## This notebook performs the following tasks:

1.) A code block that installs the nessasary packages includng PyPhi (IIT packages).


2.) A set of hard-coded functions that modify PyPhi's functionality to return the set of information loss values across all cuts, rather than just returning the minimum. *This may break in later versions of PyPhi, but works now* .

3.) Functions to generate Networks of different types at random, with corresponding TPM's.

4.) Code to generate many networks in Parallel, and record the information loss values of each cut into a .csv file.

5.) Statistical Tools for such analysis

# Section One: Generating Networks at Random using PyPhi

I reccomend using the TPU. To activate this, Select Runtime -> Change Runtime Type -> v5e-1 TPU.

I expect this line of code to run for at least an hour for large networks and/or many networks. I will also provide the data I used myself in case you do not wish to regenerate it.




In [ ]:
# Part One: Installation of Pyphi and other nessasary packages

!pip install -U "git+https://github.com/wmayner/pyphi.git@feature/iit-4.0"

import pyphi
import time
import pandas as pd
import numpy as np
import concurrent.futures

from pyphi.new_big_phi import *
from itertools import product
from pyphi import combinatorics
from pyphi.conf import config, fallback
from pyphi.direction import Direction
from pyphi.models.cuts import (
    CompleteGeneralSetPartition,
    GeneralSetPartition,
)
from pyphi.partition import system_partition_types

# ---------------------------------------------------------------------------
# PyPhi Configuration Overrides (IIT-4.0 Compatible)
# ---------------------------------------------------------------------------
# Disable internal parallelization to prevent nested process conflicts
config.PARALLEL = False

# Disable caching to prevent memory leaks during stochastic generation
config.CACHE_REPERTOIRES = False
config.CACHE_POTENTIAL_PURVIEWS = False
config.CLEAR_SUBSYSTEM_CACHES_AFTER_COMPUTING_SIA = True
config.VALIDATE_SUBSYSTEM_STATES = False

# ---------------------------------------------------------------------------
# Optimized Network Generators
# ---------------------------------------------------------------------------
def create_noisy_network(size):
    """Vectorized random network generation for speed."""
    raw_matrix = np.random.rand(2**size, size)
    state_by_state = pyphi.convert.state_by_node2state_by_state(raw_matrix)

    row_sums = state_by_state.sum(axis=1, keepdims=True)
    normalized_matrix = np.divide(
        state_by_state,
        row_sums,
        out=np.zeros_like(state_by_state),
        where=(row_sums > 0)
    )
    return pyphi.Network(normalized_matrix)

# ---------------------------------------------------------------------------
# Custom Partition Registration
# ---------------------------------------------------------------------------
@system_partition_types.register("K-PARTITIONS")
def _unidirectional_set_partitions2(node_indices, node_labels=None):
    if len(node_indices) == 1 or config.SYSTEM_PARTITION_INCLUDE_COMPLETE:
        yield CompleteGeneralSetPartition(node_indices, node_labels=node_labels)

    _node_indices = set(range(len(node_indices)))
    for partition in combinatorics.set_partitions(_node_indices, nontrivial=True):
        for d in product(Direction.all(), repeat=len(partition)):
            if Direction.CAUSE in d and Direction.EFFECT in d:
                cut_matrix = np.zeros([len(_node_indices), len(_node_indices)], dtype=int)
                for part, direction in zip(partition, d):
                    nonpart = list(_node_indices - set(part))
                    if direction == Direction.CAUSE:
                        source, target = nonpart, part
                    else:
                        source, target = part, nonpart
                    cut_matrix[np.ix_(source, target)] = 1
                    if direction == Direction.BIDIRECTIONAL:
                        cut_matrix[np.ix_(target, source)] = 1
                yield GeneralSetPartition(node_indices, cut_matrix, node_labels=node_labels, set_partition=partition)

        cut_matrix = np.zeros([len(_node_indices), len(_node_indices)], dtype=int)
        for part in partition:
            nonpart = list(_node_indices - set(part))
            cut_matrix[np.ix_(nonpart, part)] = 1
            cut_matrix[np.ix_(part, nonpart)] = 1
        yield GeneralSetPartition(node_indices, cut_matrix, node_labels=node_labels, set_partition=partition)

# ---------------------------------------------------------------------------
# Modified SIA for Early-Exit
# ---------------------------------------------------------------------------
def sia_return_all_cuts_early_exit(
    subsystem,
    repertoire_distance=None,
    directions=None,
    partition_scheme=None,
    partitions=None,
    system_state=None,
    **kwargs,
):
    partition_scheme = fallback(partition_scheme, config.SYSTEM_PARTITION_TYPE)

    if partitions is None:
        partitions = system_partitions(
            subsystem.node_indices,
            node_labels=subsystem.node_labels,
            partition_scheme=partition_scheme,
        )

    if system_state is None:
        system_state = system_intrinsic_information(subsystem, directions=directions)

    distribution = []
    # Sequential evaluation allows us to stop the moment we find a zero-phi cut
    for partition in partitions:
        sia_result = evaluate_partition(
            partition,
            subsystem=subsystem,
            system_state=system_state,
            repertoire_distance=repertoire_distance,
            directions=directions,
        )

        # Mathematical rejection: If ANY cut results in Phi <= 0, the distribution
        # doesn't meet your min(phi) > 0 criteria. Abort network immediately.
        if sia_result.phi <= 1e-10: # Using epsilon for floating point zero
            return None

        distribution.append(sia_result.phi)

    return distribution

# ---------------------------------------------------------------------------
# Worker Function for Multiprocessing
# ---------------------------------------------------------------------------
def generate_valid_distribution(size):
    """Generates networks until one satisfies min(Phi) > 0."""
    np.random.seed()
    # We must re-bind the patched function inside the worker process
    pyphi.new_big_phi.sia = sia_return_all_cuts_early_exit

    while True:
        network = create_noisy_network(size)
        state = tuple(np.random.choice([0, 1]) for _ in range(size))
        subsystem = pyphi.Subsystem(network, state)

        dist = subsystem.sia()
        if dist is not None:
            return dist

# ---------------------------------------------------------------------------
# Main Execution
# ---------------------------------------------------------------------------
if __name__ == '__main__':
    size = 7
    T = 50
    all_phi = []

    print(f"Starting parallel generation of {T} networks (N={size})...")
    start_clock = time.time()

    with concurrent.futures.ProcessPoolExecutor() as executor:
        futures = [executor.submit(generate_valid_distribution, size) for _ in range(T)]

        for i, future in enumerate(concurrent.futures.as_completed(futures), 1):
            all_phi.append(future.result())
            print(f"Completed {i}/{T} networks.", "Time_elapsed = ", time.time() - start_clock )

    # Convert to DataFrame (each network is a column)
    df = pd.DataFrame(np.array(all_phi).T, columns=[f"Phi_Dist_{i+1}" for i in range(T)])
    df.to_csv(f"phi_distributions_size{size}_T{T}.csv", index=False)

    print(f"Finished in {time.time() - start_clock:.2f} seconds.")

# Section Two: Analysis of Data

Expected Runtime: 6-ish minutes on TPU

In [5]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS",
           "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = "1"

import multiprocessing as mp
import numpy as np
import pandas as pd
from scipy.stats import genpareto
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════
# PARAMETERS
# ══════════════════════════════════════════════════════════════

CSV_FILENAME = "phi_distributions_size7_T50.csv"
CSV_OUT      = "gap_bootstrap_results.csv"

P_SAMP       = 0.05
GAMMA_GRID   = np.round(np.arange(0.01, 0.5, 0.01), 2)
T_REPS       = 500
B_BOOT       = 1000
K_MIN        = 10
K_FRAC       = 0.25
N_WORKERS    = max(1, mp.cpu_count() - 1)

# ══════════════════════════════════════════════════════════════
# GPD
# ══════════════════════════════════════════════════════════════

def fit_gpd(x_tail):
    X_k         = x_tail[-1]
    exceedances = X_k - x_tail[:-1]
    exc         = exceedances[exceedances > 1e-12]
    if len(exc) < 5:
        return 0.0, max(float(np.mean(exceedances)), 1e-12)
    try:
        c, loc, scale = genpareto.fit(exc, floc=0)
        if not np.isfinite(c) or not np.isfinite(scale) or scale <= 0:
            raise ValueError
        return float(c), float(scale)
    except Exception:
        return 0.0, max(float(np.mean(exc)), 1e-12)


def gpd_transform(X, X_k, xi, sigma):
    Y = np.maximum(X_k - X, 0.0)
    if abs(xi) < 1e-8:
        U = np.exp(-Y / sigma)
    else:
        base = np.maximum(1.0 + xi * Y / sigma, 1e-15)
        U    = base ** (-1.0 / xi)
    return np.clip(U, 1e-15, 1.0 - 1e-15)


def gpd_inverse(U, X_k, xi, sigma):
    U_safe = np.clip(U, 1e-15, 1.0 - 1e-15)
    if abs(xi) < 1e-8:
        Y = -sigma * np.log(U_safe)
    else:
        Y = (sigma / xi) * (U_safe ** (-xi) - 1.0)
    return X_k - np.maximum(Y, 0.0)

# ══════════════════════════════════════════════════════════════
# EXACT SUBSAMPLING + ROBSON
# ══════════════════════════════════════════════════════════════

def gap_bootstrap_gpd(x_samp, N_c, gammas, xi, sigma, k_idx, rng):
    n      = len(x_samp)
    X_k    = x_samp[k_idx]
    k_tail = k_idx + 1
    p      = n / N_c

    U  = gpd_transform(x_samp[:k_tail], X_k, xi, sigma)
    U1 = U[0]

    N_tail     = k_tail / p
    m_tail     = max(1, int(round(p * k_tail)))
    correction = (k_tail + 1) / (N_tail + k_tail / N_tail)

    # ══════════════════════════════════════════════════════════════
    # EXACT SUBSAMPLING (WITHOUT REPLACEMENT)
    # ══════════════════════════════════════════════════════════════
    # Vectorized extraction of m_tail distinct indices from [0, k_tail - 1]
    # rng.random generates uniform noise; argsort provides the random permutation.
    rand_weights = rng.random((B_BOOT, k_tail))
    boot_idx     = np.argsort(rand_weights, axis=1)[:, :m_tail]

    # Since U is monotonically increasing, the minimum U value in any
    # subsample corresponds perfectly to the minimum index drawn.
    U_min_boot = U[boot_idx.min(axis=1)]

    G_b   = (U_min_boot - U1) * correction
    q_U   = np.percentile(G_b, 100.0 * (1.0 - gammas))
    U_pop = U1 - q_U

    return gpd_inverse(U_pop, X_k, xi, sigma)


def robson_lower(x1, x2, gammas):
    return x1 - (1.0 / gammas - 1.0) * (x2 - x1)

# ══════════════════════════════════════════════════════════════
# WORKER
# ══════════════════════════════════════════════════════════════

def simulate_population(args):
    pop_idx, pop_array, seed = args
    rng = np.random.default_rng(seed)

    pop      = np.sort(np.unique(pop_array[np.isfinite(pop_array)]))
    M        = len(pop)
    true_min = float(pop[0])
    n        = max(10, int(round(P_SAMP * M)))
    n_g      = len(GAMMA_GRID)

    cov_gpd_sum = np.zeros(n_g)
    cov_rw_sum  = np.zeros(n_g)
    wid_gpd_sum = np.zeros(n_g)
    wid_rw_sum  = np.zeros(n_g)
    wid_gpd_ssq = np.zeros(n_g)
    wid_rw_ssq  = np.zeros(n_g)

    for t in range(T_REPS):
        idx  = rng.choice(M, size=n, replace=False)
        samp = np.sort(pop[idx])
        x1   = float(samp[0])
        x2   = float(samp[1])

        k             = min(max(K_MIN, int(K_FRAC * n)), n - 1)
        xi_hat, s_hat = fit_gpd(samp[:k + 1])

        lb_gpd = gap_bootstrap_gpd(samp, M, GAMMA_GRID, xi_hat, s_hat, k, rng)
        lb_rw  = robson_lower(x1, x2, GAMMA_GRID)

        cov_gpd = ((lb_gpd <= true_min) & (true_min <= x1)).astype(float)
        cov_rw  = ((lb_rw  <= true_min) & (true_min <= x1)).astype(float)
        wid_gpd = x1 - lb_gpd
        wid_rw  = x1 - lb_rw

        cov_gpd_sum += cov_gpd;  cov_rw_sum += cov_rw
        wid_gpd_sum += wid_gpd;  wid_rw_sum += wid_rw
        wid_gpd_ssq += wid_gpd ** 2
        wid_rw_ssq  += wid_rw  ** 2

    records = []
    for j, g in enumerate(GAMMA_GRID):
        g   = float(g)
        nom = round(1.0 - g, 4)
        for technique, cov_s, wid_s, wid_sq in [
            ("ExactGapBootstrap", cov_gpd_sum[j], wid_gpd_sum[j], wid_gpd_ssq[j]),
            ("Robson",            cov_rw_sum[j],  wid_rw_sum[j],  wid_rw_ssq[j]),
        ]:
            mean_cov = cov_s / T_REPS
            mean_wid = wid_s / T_REPS
            std_wid  = np.sqrt(max(wid_sq / T_REPS - mean_wid ** 2, 0.0))
            records.append((
                pop_idx, technique, g, nom,
                mean_cov, mean_wid, std_wid, T_REPS, P_SAMP,
            ))

    return records

# ══════════════════════════════════════════════════════════════
# PLOTS
# ══════════════════════════════════════════════════════════════

def make_plots(df):
    BG, PANEL, GRIDC = "#0d1117", "#161b22", "#21262d"
    TEXT, MUTED      = "#e6edf3", "#8b949e"
    plt.rcParams.update({
        "figure.facecolor": BG,  "axes.facecolor":  PANEL,
        "axes.edgecolor":  GRIDC, "axes.labelcolor": TEXT,
        "axes.titlecolor": TEXT,  "xtick.color":     MUTED,
        "ytick.color":     MUTED, "grid.color":      GRIDC,
        "grid.linewidth":  0.6,   "text.color":      TEXT,
        "font.family":     "monospace", "figure.dpi": 150,
        "lines.linewidth": 2.0,   "lines.markersize": 5,
    })

    agg = (df.groupby(["technique", "nominal_coverage"])
             .agg(coverage=("coverage", "mean"),
                  width=("mean_interval_length", "mean"))
             .reset_index())

    styles = {
        "ExactGapBootstrap": ("o-",  "#58a6ff", "Exact Subsampling (GPD)"),
        "Robson":            ("s--", "#f97583", "Robson-Whitlock"),
    }
    C_DIAG = "#f0e68c"
    n_pops = df["population_id"].nunique()
    title  = (f"Aggregated  ·  {n_pops} populations  ·  "
              f"p = {P_SAMP}  ·  T = {T_REPS}/pop")

    fig1, ax1 = plt.subplots(figsize=(8, 8), constrained_layout=True)
    fig1.suptitle(f"Calibration: Intended vs Actual Coverage\n{title}",
                  fontsize=11, color=TEXT)
    nominals = sorted(agg["nominal_coverage"].unique())
    lims = (min(nominals) - 0.02, 1.02)
    ax1.plot(lims, lims, color=C_DIAG, ls="--", lw=1.4,
             alpha=0.8, zorder=1, label="Perfect calibration")
    for method, (marker, color, label) in styles.items():
        sub = agg[agg["technique"] == method].sort_values("nominal_coverage")
        ax1.plot(sub["nominal_coverage"], sub["coverage"],
                 marker, color=color, alpha=0.9, label=label, zorder=3)
    ax1.set_xlim(*lims); ax1.set_ylim(*lims); ax1.set_aspect("equal")
    ticks = [0.55, 0.65, 0.75, 0.85, 0.95, 0.99]
    ax1.set_xticks(ticks); ax1.set_xticklabels([f"{t:.2f}" for t in ticks], fontsize=8)
    ax1.set_yticks(ticks); ax1.set_yticklabels([f"{t:.2f}" for t in ticks], fontsize=8)
    ax1.set_xlabel("Intended coverage  (1 − γ)", fontsize=10)
    ax1.set_ylabel("Empirical coverage", fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=10, framealpha=0.15, loc="upper left")
    fig1.savefig("calibration.png", bbox_inches="tight", facecolor=BG)
    print("Saved calibration.png")

    fig2, ax2 = plt.subplots(figsize=(8, 6), constrained_layout=True)
    fig2.suptitle(f"Efficiency Frontier\n{title}", fontsize=11, color=TEXT)
    for method, (marker, color, label) in styles.items():
        sub = agg[agg["technique"] == method].sort_values("width")
        ax2.plot(sub["width"], sub["coverage"],
                 marker, color=color, alpha=0.9, label=label, zorder=3)
    ax2.set_xlabel("Mean interval length", fontsize=10)
    ax2.set_ylabel("Empirical coverage", fontsize=10)
    ax2.set_ylim(0.50, 1.02); ax2.set_xlim(left=0)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=10, framealpha=0.15, loc="lower right")
    fig2.savefig("efficiency.png", bbox_inches="tight", facecolor=BG)
    print("Saved efficiency.png")

# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    df_in  = pd.read_csv(CSV_FILENAME)
    cols   = list(df_in.columns)
    n_pops = len(cols)

    print(f"Loaded {CSV_FILENAME}: {n_pops} populations x {len(df_in)} rows")
    print(f"  p = {P_SAMP}  T = {T_REPS}/pop  B = {B_BOOT}")
    print(f"  K_FRAC = {K_FRAC}  K_MIN = {K_MIN}")
    print(f"  gamma levels: {len(GAMMA_GRID)}  "
          f"({GAMMA_GRID[0]:.2f} to {GAMMA_GRID[-1]:.2f})")
    print(f"  total trials: {n_pops * T_REPS:,}")
    print(f"  workers: {N_WORKERS}\n")

    child_seeds = np.random.SeedSequence(42).spawn(n_pops)
    tasks = [
        (i, df_in[c].to_numpy(dtype=float), child_seeds[i])
        for i, c in enumerate(cols)
    ]

    with mp.Pool(N_WORKERS) as pool:
        all_recs = pool.map(simulate_population, tasks)

    flat = [r for sub in all_recs for r in sub]
    results_df = pd.DataFrame(flat, columns=[
        "population_id", "technique",
        "significance_level", "nominal_coverage",
        "coverage", "mean_interval_length", "std_interval_length",
        "n_trials", "p_frac",
    ])
    results_df.to_csv(CSV_OUT, index=False)
    print(f"Saved {CSV_OUT}  ({len(results_df):,} rows)")

    agg = (results_df
           .groupby(["technique", "nominal_coverage"])
           .agg(coverage=("coverage", "mean"),
                width=("mean_interval_length", "mean"))
           .reset_index())

    print(f"\n{'nominal':>8} {'cov_gpd':>9} {'cov_rw':>9} "
          f"{'wid_gpd':>10} {'wid_rw':>10}")
    print("-" * 50)
    for nom in sorted(agg["nominal_coverage"].unique()):
        sub = agg[agg["nominal_coverage"] == nom].set_index("technique")
        g = sub.loc["ExactGapBootstrap"] if "ExactGapBootstrap" in sub.index else None
        r = sub.loc["Robson"]            if "Robson"            in sub.index else None
        print(f"{nom:>8.2f} "
              f"{g['coverage'] if g is not None else float('nan'):>9.3f} "
              f"{r['coverage'] if r is not None else float('nan'):>9.3f} "
              f"{g['width']    if g is not None else float('nan'):>10.5f} "
              f"{r['width']    if r is not None else float('nan'):>10.5f}")

    make_plots(results_df)
    print("\nDone.")

Loaded phi_distributions_size7_T50.csv: 50 populations x 61888 rows
  p = 0.05  T = 500/pop  B = 1000
  K_FRAC = 0.25  K_MIN = 10
  gamma levels: 49  (0.01 to 0.49)
  total trials: 25,000
  workers: 23

Saved gap_bootstrap_results.csv  (4,900 rows)

 nominal   cov_gpd    cov_rw    wid_gpd     wid_rw
--------------------------------------------------
    0.51     0.499     0.253    0.30307    0.03913
    0.52     0.510     0.260    0.31467    0.04073
    0.53     0.521     0.269    0.32547    0.04240
    0.54     0.533     0.277    0.33650    0.04414
    0.55     0.544     0.285    0.34813    0.04595
    0.56     0.556     0.292    0.35985    0.04785
    0.57     0.567     0.300    0.37167    0.04984
    0.58     0.579     0.309    0.38424    0.05192
    0.59     0.591     0.319    0.39650    0.05411
    0.60     0.603     0.327    0.40987    0.05640
    0.61     0.614     0.338    0.42314    0.05881
    0.62     0.627     0.347    0.43794    0.06135
    0.63     0.638     0.358    0.45